In [10]:
import geopandas as gpd # 지리정보 파일 읽고 처리 # Anaconda Prompt에서 pip install geopandas로 설치해야 함
import pandas as pd # 표 형식 데이터 처리
import matplotlib.pyplot as plt # 그래프, 지도 등 시각화

In [11]:
plt.rcParams['font.family'] = 'Malgun Gothic' # 윈도우에서 한글 폰트 설정

In [12]:
prj_dir = 'C:/Users/user/Desktop/sgis-data-manual/공간분석 사례 Case Studies/Python/서울시 인구 행정동별 분석' # 'C:/SGIS/Python/서울시 행정동 인구 분석'
bord_sido = gpd.read_file(prj_dir + '/' + 'bnd_dong_11_2025_2Q.shp') # gpd.read_file(prj_dir + '/' + 'bnd_dong_11_2024_2Q.shp')
stat = pd.read_csv(prj_dir + '/' + '11_2024년_인구총괄(총인구).csv', #pd.read_csv(prj_dir + '/' + '11_2023년_인구총괄(총인구).csv',
                   names=['BASE_YEAR', 'ADM_CD', 'STAT_CD', 'POP'])

In [13]:
print(bord_sido.head()) # 기준날짜(BASE_DATE), 행정구역 이름(ADM_CD), 행정구역 코드(ADM_NM), 지리정보(geometry)로 구성

  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11010530    사직동   
1  20250630  11010540    삼청동   
2  20250630  11010550    부암동   
3  20250630  11010560    평창동   
4  20250630  11010570    무악동   

                                            geometry  
0  POLYGON ((953553.932 1953335.741, 953555.211 1...  
1  POLYGON ((953844.082 1955492.177, 953858.644 1...  
2  POLYGON ((952490.38 1956548.821, 952497.594 19...  
3  POLYGON ((953683.828 1959209.871, 953665.283 1...  
4  POLYGON ((952298.144 1953539.606, 952324.838 1...  


In [14]:
print(stat.head()) # 기준날짜(BASE_DATE), 행정구역 이름(ADM_CD), 통계 코드(STAT_NM), 인구수(POP)

   BASE_YEAR  ADM_CD    STAT_CD      POP
0       2024      11  to_in_001  9335444
1       2024      11  to_in_007  4491757
2       2024      11  to_in_008  4843687
3       2024   11010  to_in_001   144486
4       2024   11010  to_in_007    68604


In [15]:
# STAT_CD 열에 있는 고유 값 확인
unique_stat_cd = stat['STAT_CD'].unique()
print(unique_stat_cd)

['to_in_001' 'to_in_007' 'to_in_008']


In [16]:
# 총인구('to_in_001') 데이터만 필터링
stat_total = stat[stat['STAT_CD'] == 'to_in_001']

# 필터링 후 STAT_CD 고유값 재확인
unique_stat_cd = stat_total['STAT_CD'].unique()
print(unique_stat_cd)

['to_in_001']


In [17]:
stat_total = stat_total[['ADM_CD', 'POP']]

In [18]:
print(bord_sido.dtypes) # 경계(bord_sido) 각 열의 데이터 타입 확인

BASE_DATE      object
ADM_CD         object
ADM_NM         object
geometry     geometry
dtype: object


In [19]:
print(stat_total.dtypes) # 통계(stat_total) 각 열의 데이터 타입 확인

ADM_CD    int64
POP       int64
dtype: object


In [20]:
stat_total['ADM_CD'] = stat_total['ADM_CD'].astype(str) # 문자열로 변환

In [21]:
bord_sido = bord_sido.merge(stat_total, on = 'ADM_CD', how='left')
print(bord_sido.isna().sum()) # 결측값이 있는지 확인
print(bord_sido.head()) # 병합 데이터 미리보기

BASE_DATE    0
ADM_CD       0
ADM_NM       0
geometry     0
POP          0
dtype: int64
  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11010530    사직동   
1  20250630  11010540    삼청동   
2  20250630  11010550    부암동   
3  20250630  11010560    평창동   
4  20250630  11010570    무악동   

                                            geometry    POP  
0  POLYGON ((953553.932 1953335.741, 953555.211 1...   8411  
1  POLYGON ((953844.082 1955492.177, 953858.644 1...   2060  
2  POLYGON ((952490.38 1956548.821, 952497.594 19...   9057  
3  POLYGON ((953683.828 1959209.871, 953665.283 1...  16392  
4  POLYGON ((952298.144 1953539.606, 952324.838 1...   7446  


In [22]:
print(bord_sido['POP'].min(), bord_sido['POP'].max()) # 최소값과 최대값 확인하기

19 51809


In [25]:
bins = [25, 10000, 20000, 30000, 40000, 530000] # 적절한 구간 지정
labels = ['25~10,000', '10,000~10,000', '20,000~30,000', '30,000~40,000', '40,000~53,000'] # 구간의 라벨
bord_sido['POP_BINS'] = pd.cut(bord_sido['POP'], bins=bins, labels=labels, right=False)
print(bord_sido.head()) # 인구 구간('POP_BINS') 컬럼 추가 확인

  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11010530    사직동   
1  20250630  11010540    삼청동   
2  20250630  11010550    부암동   
3  20250630  11010560    평창동   
4  20250630  11010570    무악동   

                                            geometry    POP       POP_BINS  
0  POLYGON ((953553.932 1953335.741, 953555.211 1...   8411      25~10,000  
1  POLYGON ((953844.082 1955492.177, 953858.644 1...   2060      25~10,000  
2  POLYGON ((952490.38 1956548.821, 952497.594 19...   9057      25~10,000  
3  POLYGON ((953683.828 1959209.871, 953665.283 1...  16392  10,000~10,000  
4  POLYGON ((952298.144 1953539.606, 952324.838 1...   7446      25~10,000  
